In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import os
from pathlib import Path
import sys

CWD = os.getcwd()
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

In [ ]:
from data.dataset import ContrastiveDataset
from torch.utils.data import DataLoader

train_dataset = ContrastiveDataset('train')
val_dataset = ContrastiveDataset('validation')

c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def custom_collate_fn(batch):
    images = [sample[0] for sample in batch]
    targets = [sample[1] for sample in batch] 
    
    return images, targets

In [ ]:
BATCH_SIZE = 64

train_dataloader = DataLoader(train_dataset, batch_size=64, collate_fn=custom_collate_fn, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, collate_fn=custom_collate_fn)

In [ ]:
from src.modules.clip import CLIP
import torch.optim as optim

LEARNING_RATE = 1e-3
EPOCH_SIZE = 10
EVAL_ITER = 2
clip_model = CLIP(device=device).to(device)

decay    = [p for n, p in clip_model.named_parameters() if p.requires_grad and n != "logit_scale"]
no_decay = [clip_model.logit_scale]

optimizer = optim.AdamW(
    [{"params": decay, "weight_decay": 0.01},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE,
)

In [ ]:
import torch.nn.functional as F
import numpy as np
import math

for epoch in range(EPOCH_SIZE):
    total_train_loss = 0
    total_val_loss = 0
    for image_batch, text_batch in train_dataloader:
        optimizer.zero_grad()
        n = len(image_batch)
        logits = clip_model(image_batch, text_batch)
        label = torch.arange(n).to(device)
        loss_image = F.cross_entropy(logits, label)
        loss_text = F.cross_entropy(logits.T, label)
        loss = (loss_image+loss_text)/2
        total_train_loss += loss
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            clip_model.logit_scale.clamp_(0, math.log(100))
    del image_batch
    del text_batch
    if epoch % EVAL_ITER == 0 or epoch == EPOCH_SIZE - 1:
        clip_model.eval() 
        with torch.no_grad():
            for image_batch, text_batch in val_dataloader:
                n = len(image_batch)
                logits = clip_model(image_batch, text_batch)
                label = torch.arange(n).to(device)
                loss_image = F.cross_entropy(logits, label)
                loss_text = F.cross_entropy(logits.T, label)
                loss = (loss_image+loss_text)/2
                total_val_loss += loss
            del image_batch
            del text_batch
        train_avg_loss = total_train_loss / len(train_dataloader) 
        val_avg_loss = total_val_loss / len(val_dataloader) 
        print(f"Epoch [{epoch + 1}/{EPOCH_SIZE}], Train Loss: {train_avg_loss.item():.4f}, Val Loss: {val_avg_loss.item():.4f}")
    clip_model.train() 



In [ ]:
trainable_names = {n for n, p in clip_model.named_parameters() if p.requires_grad}
head_state = {k: v for k, v in clip_model.state_dict().items() if k in trainable_names}

torch.save({
    "model": head_state,
    "config": {"image_model_name": "vit-base-16",
               "text_model_name": "qwen3-0.6B",
               "output_size": 64},
    "optimizer": optimizer.state_dict(),
    "epoch": epoch,
}, "clip_head.pt")
